In [37]:
using JuMP
using Gurobi
using MathOptInterface
# Model
model = Model(Gurobi.Optimizer)

# Parameters
hours = 4
buses = 2
generators = 2
lines = 1
candidate_lines = 4
cand_lines = 1

# Generator data
gen_cost = [10, 12]  # $/MWh
gen_capacity = [1000, 100]  # MW
gen_ramp_rate = [100, 10]  # MW/min
gen_bus = [1, 2]

# Load data
load = [[0,0,0,0],[225, 250, 180, 160]]  # MW (load at bus 2)

# Transmission line data
line_capacity = [100]  # MW
line_reactance = [0.05917]  # p.u.
line_buses = [(1, 2)]

BigM = 5.0*candidate_capacity*candidate_lines

# Candidate line data
candidate_capacity = 25  # MW
candidate_reactance = 0.05917  # p.u.
candidate_cost = 10  # $/MW
candidate_buses = [(1, 2)]

# Variables
@variable(model, g[1:generators, 1:hours])  # Generator output
@variable(model, -line_capacity[1] <= f[1:lines, 1:hours] <= line_capacity[1])  # Line flow
@variable(model, x[1:cand_lines, 1:hours])  # Candidate line flows
@variable(model, 0 <= cand_num <= candidate_lines, Int)  # Line flow
@variable(model, theta[1:buses, 1:hours])  # Bus voltage angles

# Objective: Minimize generation cost + expansion cost
@objective(model, Min, sum(gen_cost[i] * g[i, t] for i in 1:generators, t in 1:hours) +
						candidate_cost * cand_num * candidate_capacity)

# Constraints
for t in 1:hours
	# Power balance at each bus
	for b in 1:buses
		@constraint(model, sum(g[i, t] for i in 1:generators if gen_bus[i] == b) +
							sum(f[l, t] for l in 1:lines if line_buses[l][2] == b) -
							sum(f[l, t] for l in 1:lines if line_buses[l][1] == b) +
							sum(x[l, t] for l in 1:cand_lines if candidate_buses[l][2] == b) -
							sum(x[l, t] for l in 1:cand_lines if candidate_buses[l][1] == b) ==
							(b == 2 ? load[b][t] : 0))
	end
	
	for i in 1:generators
		@constraint(model, 0 <= g[i, t] <= gen_capacity[i])
	end

	# Line flow constraints
	for l in 1:lines
		@constraint(model, f[l, t] == (theta[line_buses[l][1], t] - theta[line_buses[l][2], t]) / line_reactance[l])
	end

	# Candidate line flow constraints
	for l in 1:candidate_lines
		@constraint(model, BigM*cand_num+x[1,t]-((l*(theta[candidate_buses[1][1], t] - theta[candidate_buses[1][2], t])) / candidate_reactance) <= BigM*l)
		@constraint(model, x[1,t]-((l*(theta[candidate_buses[1][1], t] - theta[candidate_buses[1][2], t])) / candidate_reactance)-BigM*cand_num >= -BigM*l)
	end

	@constraint(model, x[1,t] - cand_num*candidate_capacity <= 0)
	@constraint(model, x[1,t] + cand_num*candidate_capacity >= 0)

	# Reference bus angle
	@constraint(model, theta[1, t] == 0)

end

for t in 2:hours
	for i in 1:generators
		@constraint(model, -gen_ramp_rate[i] <= g[i, t] - g[i, t-1] <= gen_ramp_rate[i])
	end
end

output_path = "/Users/sc87/code/OPF_LASCOPF_Staple/Horizontal_Proper/simplest_problem.lp"
write_to_file(model, output_path)
println("LP file has been created at: $output_path")

# Solve the model
optimize!(model)

list_of_conflicting_constraints = ConstraintRef[]
if get_attribute(model, MOI.ConflictStatus()) == MOI.CONFLICT_FOUND
        for (F, S) in list_of_constraint_types(model)
                for con in all_constraints(model, F, S)
                    if get_attribute(con, MOI.ConstraintConflictStatus()) == MOI.IN_CONFLICT
                        push!(list_of_conflicting_constraints, con)
                    end
                end
        end
        display(list_of_conflicting_constraints)
        return model, list_of_conflicting_constraints
else
        @info "Conflicts computation failed."
        display(list_of_conflicting_constraints)
end

# Results
println("Objective value: ", objective_value(model))
println("Generator outputs: ", value.(g))
println("Line flows: ", value.(f))
println("Candidate line capacities: ", value.(x))



ConstraintRef[]

Set parameter Username
Academic license - for non-commercial use only - expires 2026-01-23
LP file has been created at: /Users/sc87/code/OPF_LASCOPF_Staple/Horizontal_Proper/simplest_problem.lp
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[x86] - Darwin 24.3.0 24D70)

CPU model: Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 70 rows, 39 columns and 218 nonzeros
Model fingerprint: 0x0925f722
Variable types: 38 continuous, 1 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+02]
  Objective range  [1e+01, 2e+02]
  Bounds range     [4e+00, 1e+03]
  RHS range        [2e+02, 2e+03]
Presolve removed 16 rows and 16 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 12 available processors)

Solution count 0

Model is infeasible or unbounded
Best objective -, best bound -, gap -

User-callback 

┌ Info: Conflicts computation failed.
└ @ Main /Users/sc87/code/OPF_LASCOPF_Staple/Horizontal_Proper/Example/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W0sZmlsZQ==.jl:108


MathOptInterface.ResultIndexBoundsError{MathOptInterface.ObjectiveValue}: Result index of attribute MathOptInterface.ObjectiveValue(1) out of bounds. There are currently 0 solution(s) in the model.